# K-means TC4 Fashion-MNIST: tái chạy trên Colab bằng Julia

Notebook độc lập để tái chạy một lượt fit codebook K=32, seed=123 bằng chính pipeline Julia của dự án. Chọn runtime **Julia** và CPU trong Colab; mọi ô code chạy trực tiếp trong Julia kernel, không có Python kernel hay Python gọi Julia.

Notebook tải đúng commit của repo, xác minh bốn archive Fashion-MNIST, rồi chạy TC1–TC3 để tạo descriptor theo đúng mã nguồn hiện có và chạy đúng một fit tại TC4. Phạm vi fit gồm cả 70.000 ảnh như cấu hình phát hành; nhãn không được dùng để tạo descriptor hoặc fit.

Profile phát hành giới hạn K-means ở 300 vòng. Notebook ghi đè riêng giới hạn lên 400 vòng; đây là mức trần, không ép số vòng hội tụ. CPU, phiên bản Julia và thư viện số học có thể làm số vòng thực tế khác nhau.


In [ ]:
# 00 · Khởi tạo runtime Julia CPU
# Chọn Runtime → Change runtime type → Runtime type: Julia, Hardware accelerator: None.
# JULIA_NUM_THREADS phải được đặt trước khi kernel khởi động; ở đây ta ghi cấu hình
# mong muốn và luôn in số thread thực tế của kernel Colab.
ENV["JULIA_NUM_THREADS"] = "1"
ENV["OPENBLAS_NUM_THREADS"] = "1"
ENV["OPENBLAS_CORETYPE"] = "Haswell"
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = "0"

using Pkg, SHA, Markdown, LibGit2
using LinearAlgebra

const REPOSITORY_URL = "https://github.com/NguyenVoTanDuy2006/DataMining-FIM-Project.git"
const SOURCE_BRANCH = "feature/fashion-mnist-consolidated"
const SOURCE_COMMIT = "dbadabc4af05070c602b3bcd890150fcd496c885"
const REPO_ROOT = joinpath("/content", "fashion-fim-" * first(SOURCE_COMMIT, 8))
const DEPOT_ROOT = joinpath("/content", "fashion-demo-work", "julia_depot")
mkpath(DEPOT_ROOT)
DEPOT_ROOT in DEPOT_PATH || pushfirst!(DEPOT_PATH, DEPOT_ROOT)
ENV["JULIA_DEPOT_PATH"] = join(DEPOT_PATH, ":")

git_repo = if !ispath(REPO_ROOT)
    mkpath(dirname(REPO_ROOT))
    LibGit2.clone(REPOSITORY_URL, REPO_ROOT; branch=SOURCE_BRANCH)
elseif isdir(joinpath(REPO_ROOT, ".git"))
    LibGit2.GitRepo(REPO_ROOT)
else
    error("$REPO_ROOT exists but is not the pinned Git repository. Restart the Colab Julia runtime.")
end
actual_commit = try
    LibGit2.checkout!(git_repo, SOURCE_COMMIT; force=true)
    string(LibGit2.head_oid(git_repo))
finally
    close(git_repo)
end
actual_commit == SOURCE_COMMIT || error("Expected source $SOURCE_COMMIT, found $actual_commit. Restart the runtime.")
ENV["FASHION_DEMO_REPO_ROOT"] = REPO_ROOT
ENV["FASHION_DEMO_SOURCE_COMMIT"] = SOURCE_COMMIT

Pkg.activate(REPO_ROOT)
Pkg.instantiate()
using EclatFIM
using EclatFIM.FashionMNIST: FashionTC
BLAS.set_num_threads(1)

struct FashionDemoPNG
    bytes::Vector{UInt8}
end
Base.show(io::IO, ::MIME"image/png", image::FashionDemoPNG) = write(io, image.bytes)
Base.showable(::MIME"image/png", ::FashionDemoPNG) = true

function show_demo_markdown(path)
    display(Markdown.parse(read(path, String)))
end
function show_demo_image(path; mime="image/png", alt=basename(path))
    display(Markdown.parse("**$alt**"))
    if mime == "image/png"
        display(MIME(mime), FashionDemoPNG(read(path)))
    elseif mime == "image/svg+xml"
        display(MIME(mime), read(path, String))
    else
        error("Unsupported image MIME type: $mime")
    end
end

println("Source commit: ", actual_commit)
println("Colab Julia runtime: ", VERSION)
println("Active Julia threads: ", Threads.nthreads(), " (fixed when the kernel started)")
println("OpenBLAS threads: ", BLAS.get_num_threads(), "; requested Haswell target: ", ENV["OPENBLAS_CORETYPE"])
println("Julia depot: ", DEPOT_ROOT)
println("EclatFIM is ready in this native Julia kernel.")


## 1. Cấu hình và phạm vi fit

Patch 4×4 không chồng lấp; giữ patch nếu có pixel > 29; descriptor raw pixel 16 chiều, Float32/255 và column-major trong patch; không sampling; K=32; seed=123; K-means++; squared Euclidean; dung sai 1e-4. Notebook thay `max_iterations` thành 400 trong một bản cấu hình tạm, không sửa `release.toml`.

## 2. Tạo descriptor và fit bằng FashionTC

TC1–TC3 chạy trực tiếp bằng hàm Julia hiện có để tạo descriptor trên cả train và test. TC4 chạy một lần fit. Notebook lưu tâm cụm CSV và `run_summary.toml` dưới `/content/fashion_tc4_kmeans_reproduction/output`.

## 3. Đọc kết quả

Đọc số vòng, trạng thái hội tụ, thời gian, số cụm có phần tử, entropy, phiên bản Julia và số luồng thực tế từ phần output và config snapshot và run summary. `JULIA_NUM_THREADS` phải được đặt trước khi kernel khởi động; ô khởi tạo ghi giá trị yêu cầu nhưng cũng lưu số luồng thực tế đang hoạt động.


In [ ]:
using EclatFIM
using EclatFIM.FashionMNIST: FashionTC, create_dataset_manifest,
    download_fashion_mnist, new_release_run_id
using LinearAlgebra, TOML

repo_root = ENV["FASHION_DEMO_REPO_ROOT"]
source_commit = ENV["FASHION_DEMO_SOURCE_COMMIT"]
data_dir = joinpath(repo_root, "data", "raw", "fashion-mnist")
release_profile = joinpath(repo_root, "src", "applications", "Fashion-Mnist",
    "configs", "transactions", "release.toml")
output_dir = "/content/fashion_tc4_kmeans_reproduction/output"
mkpath(output_dir)

println("Downloading or verifying the official Fashion-MNIST archives...")
download_fashion_mnist(data_dir)
run_id = new_release_run_id()
run_root = joinpath(repo_root, "results", "fashion", "release", run_id)
dataset = create_dataset_manifest(data_dir, run_root; repo_root, run_id, download_missing=false)
println("Dataset run: ", run_id)
println("Verified images: train=", length(dataset.train.labels), ", test=", length(dataset.test.labels))

profile = TOML.parsefile(release_profile)
profile["run"]["run_id"] = run_id
profile["tc4_codebook"]["max_iterations"] = 400
profile["execution"]["backend"] == "cpu" || error("This notebook requires CPU execution.")
config_dir = mktempdir(prefix="fashion-tc4-config-")
config_path = joinpath(config_dir, "release.toml")
open(config_path, "w") do io
    TOML.print(io, profile)
end
state = try
    FashionTC.prepare_run(config_path; repo_root, data_dir, dataset_manifest_path=dataset.manifest_path)
finally
    rm(config_dir; recursive=true, force=true)
end

for (step, stage) in enumerate((FashionTC.run_tc1!, FashionTC.run_tc2!,
        FashionTC.run_tc3!, FashionTC.run_tc4!))
    println("\nRunning source pipeline stage TC$(step)...")
    println(stage(state))
end

length(state.run_ids) == 1 || error("Expected exactly one K=32, seed=123 codebook fit.")
tc_run_id = only(state.run_ids)
fit_dir = state.run_dirs[tc_run_id]
metrics = TOML.parsefile(joinpath(fit_dir, "tc4_metrics.toml"))
codebook = FashionTC._load_codebook(joinpath(fit_dir, "codebook.bin"))
centroids = codebook.centroids
size(centroids) == (16, 32) || error("Expected 32 centroids with 16 dimensions each.")
size(state.descriptor_matrix, 2) == 2_143_746 || error("Unexpected retained descriptor count.")

centroids_path = joinpath(output_dir, "centroids.csv")
open(centroids_path, "w") do io
    println(io, "cluster_id,", join(["pixel_" * string(i) for i in 1:16], ","))
    for cluster in axes(centroids, 2)
        println(io, cluster, ",", join(string.(centroids[:, cluster]), ","))
    end
end

snapshot = deepcopy(profile)
snapshot["runtime"] = Dict{String,Any}(
    "julia_version" => string(VERSION),
    "julia_threads_active" => Threads.nthreads(),
    "julia_threads_requested" => 1,
    "openblas_threads" => BLAS.get_num_threads(),
    "openblas_coretype_requested" => ENV["OPENBLAS_CORETYPE"],
)
snapshot_path = joinpath(output_dir, "config_snapshot.toml")
open(snapshot_path, "w") do io
    TOML.print(io, snapshot)
end

summary = Dict{String,Any}(
    "status" => metrics["status"],
    "run_id" => run_id,
    "tc_run_id" => tc_run_id,
    "config_snapshot" => snapshot_path,
    "source_commit" => source_commit,
    "dataset" => "Fashion-MNIST",
    "train_images" => length(dataset.train.labels),
    "test_images" => length(dataset.test.labels),
    "fit_scope" => "all retained descriptors from 70,000 images",
    "labels_used_for_fit" => false,
    "descriptor_count" => size(state.descriptor_matrix, 2),
    "descriptor_dimensions" => 16,
    "k" => 32,
    "seed" => 123,
    "initialization" => metrics["initialization"],
    "distance" => metrics["distance"],
    "max_iterations" => metrics["max_iterations"],
    "tolerance" => metrics["tolerance"],
    "iterations" => metrics["iterations"],
    "converged" => metrics["converged"],
    "stop_reason" => metrics["stop_reason"],
    "fit_seconds" => metrics["fit_seconds"],
    "distortion_mean_squared" => metrics["distortion_mean_squared"],
    "cluster_occupancy_used" => metrics["occupancy_used"],
    "normalized_entropy" => metrics["normalized_entropy"],
    "julia_version" => string(VERSION),
    "julia_threads_active" => Threads.nthreads(),
    "julia_threads_requested" => 1,
    "openblas_threads" => BLAS.get_num_threads(),
    "openblas_coretype_requested" => ENV["OPENBLAS_CORETYPE"],
    "clustering_version" => metrics["library_version"],
)
summary_path = joinpath(output_dir, "run_summary.toml")
open(summary_path, "w") do io
    TOML.print(io, summary)
end

println("\nTC4 status: ", summary["status"])
println("K-means iterations: ", summary["iterations"], " / ", summary["max_iterations"])
println("Converged: ", summary["converged"])
println("Fit seconds: ", round(summary["fit_seconds"]; digits=2))
println("Occupied clusters: ", summary["cluster_occupancy_used"], " / ", summary["k"])
println("Normalized entropy: ", summary["normalized_entropy"])
println("Julia: ", summary["julia_version"], " | active threads: ", summary["julia_threads_active"])
println("OpenBLAS threads: ", summary["openblas_threads"], " | requested core target: ", summary["openblas_coretype_requested"])
println("Config snapshot: ", snapshot_path)
println("Summary: ", summary_path)
println("Centroids: ", centroids_path)
!summary["converged"] && println("K-means reached its configured limit without convergence; status is preserved in run_summary.toml.")
